# Trace quality: does the answer appear in the reasoning — and on the page?

**Questions**
1. Do the numbers from the model's final answer already show up in the reasoning trace?
2. *How early* in the trace is the first hit?
3. Do those same numbers also appear in the retrieved page text (a simple grounding check)?

**Method.** Regex / string search over number variants (`772607`, `772,607`, `772.607`, …).
No LLM judge. This is a first base check — false friends are possible (same digits, different meaning).

Uses `analysis/derived/page_outcomes.parquet` and `pages_*.parquet` from `run_comparison.ipynb`.
Includes all five runs (gpt-oss low/medium/high, DeepSeek high/max).

## 0. Setup

In [8]:
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 200)

ROOT = Path.cwd().resolve()
if not (ROOT / "output").is_dir():
    ROOT = ROOT.parent

DERIVED = ROOT / "analysis" / "derived"
SEED = 42
KEEP_RUNS = [
    "gpt-oss low", "gpt-oss medium", "gpt-oss high",
    "DeepSeek high", "DeepSeek max",
]

## 1. Load page-level traces

We only keep calls with a present reasoning trace and a valid JSON answer that contains at least one KPI value.
Empty answers (`KPI_Entries: []`) have nothing to look for.

In [9]:
pages = pd.read_parquet(DERIVED / "page_outcomes.parquet")
pages = pages[pages.run.isin(KEEP_RUNS) & (pages.trace_state == "present")].copy()


def parse_entries(text):
    if not isinstance(text, str) or not text.strip():
        return []
    t = text.strip()
    fenced = re.fullmatch(r"```(?:json)?\s*(.*?)\s*```", t, flags=re.S)
    if fenced:
        t = fenced.group(1)
    try:
        data = json.loads(t)
    except ValueError:
        return []
    entries = data.get("KPI_Entries") if isinstance(data, dict) else None
    if not isinstance(entries, list):
        return []
    return entries


def entry_values(entries):
    vals = []
    for e in entries:
        try:
            vals.append(float(str(e.get("value")).replace(",", "")))
        except (TypeError, ValueError, AttributeError):
            continue
    return vals


pages["entries"] = pages["answer"].map(parse_entries)
pages["answer_values"] = pages["entries"].map(entry_values)
with_vals = pages[pages["answer_values"].map(len) > 0].copy()

print(f"{len(pages):,} traces  |  {len(with_vals):,} with ≥1 answer value")
display(with_vals.groupby("run", observed=True).size().rename("calls with values").to_frame())

3,826 traces  |  447 with ≥1 answer value


,calls with values
run,
DeepSeek high,107
DeepSeek max,60
gpt-oss high,53
gpt-oss low,115
gpt-oss medium,112


## 2. Number matching helpers

A value like `772607` can appear as `772607`, `772,607`, `772.607` or `772 607`.
We search for those spellings and reject hits that sit inside a longer digit run
(so `772607` does not match inside `17726071`).

In [10]:
def number_spellings(value):
    try:
        x = float(value)
    except (TypeError, ValueError):
        return []
    if not np.isfinite(x):
        return []
    as_int = abs(x - round(x)) < 1e-9
    out = set()
    if as_int:
        n = int(round(x))
        s = str(n)
        out.add(s)
        out.add(f"{n:,}")                 # 772,607
        out.add(f"{n:,}".replace(",", " "))
        out.add(f"{n:,}".replace(",", "."))
        # European thousands + decimal not needed for ints
    else:
        out.add(str(x))
        out.add(f"{x:.1f}")
        out.add(f"{x:.2f}")
        out.add(f"{x:.3f}".rstrip("0").rstrip("."))
    return sorted(out, key=len, reverse=True)


def first_match(text, value):
    """Return (start, matched_string) for the earliest safe hit, or None."""
    if not isinstance(text, str) or not text:
        return None
    best = None
    for spelling in number_spellings(value):
        for m in re.finditer(re.escape(spelling), text):
            start, end = m.start(), m.end()
            left = text[start - 1] if start else ""
            right = text[end] if end < len(text) else ""
            if left.isdigit() or right.isdigit():
                continue
            if best is None or start < best[0]:
                best = (start, spelling)
            break
    return best


def match_report(text, values):
    hits = [first_match(text, v) for v in values]
    starts = [h[0] for h in hits if h is not None]
    return {
        "n_values": len(values),
        "n_found": len(starts),
        "all_found": len(starts) == len(values) and len(values) > 0,
        "any_found": len(starts) > 0,
        "first_char": min(starts) if starts else np.nan,
    }

## 3. Build match table (all calls)

For every call we record:
- whether the final answer contains KPI numbers
- whether those numbers appear in the reasoning / on the page
- page outcome (`category`)

Empty answers have no answer-numbers to search; for them we instead check whether
**any number from the page text** also appears in the trace ("talked about page numbers but answered empty").

In [ ]:
def match_values(text, values):
    hits = [first_match(text, v) for v in values]
    starts = [h[0] for h in hits if h is not None]
    return {
        "n_values": len(values),
        "n_found": len(starts),
        "all_found": bool(values) and len(starts) == len(values),
        "any_found": len(starts) > 0,
        "first_char": min(starts) if starts else np.nan,
    }


def slug(name: str) -> str:
    return re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_").lower()


page_text_cache = {}
for run in KEEP_RUNS:
    path = DERIVED / f"pages_{slug(run)}.parquet"
    if not path.is_file():
        print(f"missing page cache: {path.name}")
        continue
    page_text_cache[run] = (
        pd.read_parquet(path)
          .assign(page_label=lambda d: d["page_label"].astype(str))
          .drop_duplicates(["report", "page_label"])
          .set_index(["report", "page_label"])["page_text"]
    )


def lookup_page_text(run, report, page):
    series = page_text_cache.get(run)
    if series is None or page is None or (isinstance(page, float) and np.isnan(page)):
        return None
    return series.get((report, str(page)))


# Numbers that look like KPI candidates on a page (avoid tiny ints like years alone by
# also keeping years; we only use them for the empty-answer echo check).
PAGE_NUM = re.compile(r"(?<!\d)(\d{1,3}(?:[ ,.]\d{3})+|\d{4,})(?!\d)")


def page_numbers(text: str, limit: int = 40) -> list[float]:
    if not text:
        return []
    vals = []
    seen = set()
    for tok in PAGE_NUM.findall(text):
        for spelling_as_float in (
            tok.replace(",", "").replace(" ", ""),
            tok.replace(".", "").replace(",", "").replace(" ", ""),
            tok.replace(".", "").replace(",", ".").replace(" ", ""),
        ):
            try:
                v = float(spelling_as_float)
            except ValueError:
                continue
            if v in seen:
                continue
            seen.add(v)
            vals.append(v)
            break
        if len(vals) >= limit:
            break
    return vals


rows = []
for row in pages.itertuples():
    values = row.answer_values
    has_answer_values = len(values) > 0
    page_text = lookup_page_text(row.run, row.report, row.page) or ""
    has_page_text = bool(page_text)

    if has_answer_values:
        tinfo = match_values(row.reasoning, values)
        pinfo = match_values(page_text, values)
        echo = None  # only defined for empty answers
    else:
        tinfo = dict(n_values=0, n_found=0, all_found=False, any_found=False, first_char=np.nan)
        pinfo = dict(n_values=0, n_found=0, all_found=False, any_found=False, first_char=np.nan)
        # page numbers mentioned in the trace despite empty answer
        echo_vals = page_numbers(page_text)
        echo = match_values(row.reasoning, echo_vals)["any_found"] if echo_vals else False

    rows.append({
        "run": row.run,
        "report": row.report,
        "page": row.page,
        "category": row.category,
        "n_gold_page": row.n_gold_page,
        "has_answer_values": has_answer_values,
        "has_page_text": has_page_text,
        "trace_chars": len(row.reasoning or ""),
        "answer_in_trace_all": tinfo["all_found"],
        "answer_in_trace_any": tinfo["any_found"],
        "answer_on_page_all": pinfo["all_found"],
        "answer_on_page_any": pinfo["any_found"],
        "first_char": tinfo["first_char"],
        "page_number_echoed_in_trace": echo,
        "n_answer_values": tinfo["n_values"],
    })

df = pd.DataFrame(rows)
df["first_frac"] = df["first_char"] / df["trace_chars"].replace(0, np.nan)
df["run"] = pd.Categorical(df["run"], KEEP_RUNS, ordered=True)

# Convenient subsets
ans = df[df.has_answer_values].copy()
empty = df[~df.has_answer_values].copy()

print(f"all calls: {len(df):,}  |  with answer values: {len(ans):,}  |  empty answers: {len(empty):,}")
display(df.groupby("run", observed=True).agg(
    calls=("report", "size"),
    with_answer_values=("has_answer_values", "sum"),
    empty_answers=("has_answer_values", lambda s: (~s).sum()),
))

## 4. Overview plots

All bars are **shares within a run** (comparable across runs). Absolute `n` is only annotated.

In [ ]:
RUN_COLORS = dict(zip(KEEP_RUNS, plt.cm.tab10.colors[: len(KEEP_RUNS)]))


def share_bar(ax, table, title, ylabel="share of calls"):
    """table: index=run, columns=categories, values=shares that sum ~1."""
    table = table.reindex(KEEP_RUNS)
    table.plot(kind="bar", stacked=True, ax=ax, width=0.85)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("")
    ax.set_ylim(0, 1)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=2, frameon=False)
    ax.tick_params(axis="x", rotation=30)


# --- 4.1 Answer present? ---
ans_mix = (
    df.assign(answer=np.where(df.has_answer_values, "has KPI number(s)", "empty KPI_Entries"))
      .groupby(["run", "answer"], observed=True).size()
      .groupby("run", observed=True).transform(lambda s: s / s.sum())
      .unstack("answer")
      .fillna(0)
)
# stable column order
ans_mix = ans_mix.reindex(columns=["has KPI number(s)", "empty KPI_Entries"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
share_bar(axes[0], ans_mix, "Calls with vs without answer numbers")
ns = df.groupby("run", observed=True).size().reindex(KEEP_RUNS)
axes[0].set_xticklabels([f"{r}\n(n={ns[r]})" for r in KEEP_RUNS])

# --- 4.2 If there is an answer number: is it in the trace? ---
trace_mix = (
    ans.assign(status=np.select(
        [ans.answer_in_trace_all, ans.answer_in_trace_any],
        ["all answer values in trace", "some (not all) in trace"],
        default="no answer value in trace",
    ))
    .groupby(["run", "status"], observed=True).size()
    .groupby("run", observed=True).transform(lambda s: s / s.sum())
    .unstack("status")
    .fillna(0)
)
trace_mix = trace_mix.reindex(columns=[
    "all answer values in trace", "some (not all) in trace", "no answer value in trace"
])
share_bar(axes[1], trace_mix, "Among calls WITH answer numbers: in trace?")
ns_ans = ans.groupby("run", observed=True).size().reindex(KEEP_RUNS)
axes[1].set_xticklabels([f"{r}\n(n={ns_ans[r]})" for r in KEEP_RUNS])
plt.tight_layout()
plt.show()

In [ ]:
# --- 4.3 Correct vs wrong, depending on whether answer numbers are in the trace ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

def cat_mix(sub, title, ax):
    tab = (
        sub.groupby(["run", "category"], observed=True).size()
           .groupby("run", observed=True).transform(lambda s: s / s.sum())
           .unstack("category").fillna(0)
    )
    # prefer a stable order if present
    cols = [c for c in [
        "correct, no gold values", "correct, gold values", "error"
    ] if c in tab.columns] + [c for c in tab.columns if c not in {
        "correct, no gold values", "correct, gold values", "error"
    }]
    share_bar(ax, tab[cols], title)
    n = sub.groupby("run", observed=True).size().reindex(KEEP_RUNS)
    ax.set_xticklabels([f"{r}\n(n={n.get(r, 0)})" for r in KEEP_RUNS])

in_trace = ans[ans.answer_in_trace_any]
not_in_trace = ans[~ans.answer_in_trace_any]
cat_mix(in_trace, "Outcome when answer number IS in trace", axes[0])
cat_mix(not_in_trace, "Outcome when answer number NOT in trace", axes[1])
plt.tight_layout()
plt.show()

print("Calls with answer values but number never appears in trace:")
display(not_in_trace.groupby(["run", "category"], observed=True).size().unstack("category").fillna(0).astype(int))

In [ ]:
# --- 4.4 How early in the trace? (only where at least one answer value was found) ---
early = ans[ans.answer_in_trace_any].dropna(subset=["first_frac"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for run, g in early.groupby("run", observed=True):
    axes[0].hist(
        g["first_frac"], bins=20, range=(0, 1), density=True, alpha=0.4,
        color=RUN_COLORS[run], label=f"{run} (n={len(g)})",
    )
axes[0].set_xlabel("first hit as fraction of trace length")
axes[0].set_ylabel("density")
axes[0].set_title("How early? (normalized per run)")
axes[0].legend(fontsize=8)

# box/median view is easier to compare across 5 runs
order = [r for r in KEEP_RUNS if r in set(early["run"])]
data = [early.loc[early.run == r, "first_frac"] for r in order]
axes[1].boxplot(data, tick_labels=order, vert=True, showfliers=False)
axes[1].set_ylabel("first_frac")
axes[1].set_title("Early-hit distribution (boxplot)")
axes[1].tick_params(axis="x", rotation=30)
axes[1].set_ylim(0, 1)
plt.tight_layout()
plt.show()

display(early.groupby("run", observed=True)["first_frac"].agg(["count", "median", "mean"]).round(3))

## 5. Grounding: are answer numbers on the retrieved page?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

page_mix = (
    ans.assign(status=np.select(
        [ans.answer_on_page_all, ans.answer_on_page_any],
        ["all answer values on page", "some (not all) on page"],
        default="no answer value on page",
    ))
    .groupby(["run", "status"], observed=True).size()
    .groupby("run", observed=True).transform(lambda s: s / s.sum())
    .unstack("status").fillna(0)
)
page_mix = page_mix.reindex(columns=[
    "all answer values on page", "some (not all) on page", "no answer value on page"
])
share_bar(axes[0], page_mix, "Among calls WITH answer numbers: on page?")
axes[0].set_xticklabels([f"{r}\n(n={ns_ans[r]})" for r in KEEP_RUNS])

# Trace × page buckets
bucket = np.select(
    [
        ans.answer_in_trace_all & ans.answer_on_page_all,
        ans.answer_in_trace_any & ~ans.answer_on_page_any,
        ~ans.answer_in_trace_any & ans.answer_on_page_any,
        ~ans.answer_in_trace_any & ~ans.answer_on_page_any,
        ans.answer_in_trace_any & ans.answer_on_page_any,
    ],
    [
        "in trace + on page (all)",
        "in trace, NOT on page",
        "on page, NOT in trace",
        "neither",
        "partial overlap",
    ],
    default="partial overlap",
)
ans = ans.copy()
ans["bucket"] = bucket
buck = (
    ans.groupby(["run", "bucket"], observed=True).size()
       .groupby("run", observed=True).transform(lambda s: s / s.sum())
       .unstack("bucket").fillna(0)
)
share_bar(axes[1], buck, "Trace vs page (answer numbers)")
axes[1].set_xticklabels([f"{r}\n(n={ns_ans[r]})" for r in KEEP_RUNS])
plt.tight_layout()
plt.show()

In [ ]:
# Outcome by grounding bucket (stacked shares within each run×bucket would be busy —
# show: among "in trace + on page" vs "in trace, NOT on page", what share are errors?)
focus = ans[ans.bucket.isin(["in trace + on page (all)", "in trace, NOT on page"])].copy()
err_share = (
    focus.assign(is_error=focus.category.eq("error"))
         .groupby(["run", "bucket"], observed=True)["is_error"].mean()
         .unstack("bucket")
)
fig, ax = plt.subplots(figsize=(8, 4))
err_share.reindex(KEEP_RUNS).plot(kind="bar", ax=ax)
ax.set_ylabel("share errors")
ax.set_xlabel("")
ax.set_title("Error rate: answer numbers grounded on page vs only in trace")
ax.set_ylim(0, 1)
ax.legend(title="bucket")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()
display(err_share.reindex(KEEP_RUNS).style.format("{:.1%}"))

## 6. Empty answers: number in the trace anyway?

These calls end with `KPI_Entries: []`. We ask:
1. Was there gold on this page (`n_gold_page > 0`) — i.e. a missed extraction?
2. Do numbers from the page text still appear in the reasoning (page-number echo)?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

empty_gold = (
    empty.assign(gold=np.where(empty.n_gold_page > 0, "empty answer, gold on page", "empty answer, no gold"))
         .groupby(["run", "gold"], observed=True).size()
         .groupby("run", observed=True).transform(lambda s: s / s.sum())
         .unstack("gold").fillna(0)
)
share_bar(axes[0], empty_gold, "Empty answers: was there gold on the page?")
ns_empty = empty.groupby("run", observed=True).size().reindex(KEEP_RUNS)
axes[0].set_xticklabels([f"{r}\n(n={ns_empty[r]})" for r in KEEP_RUNS])

# Among empty answers that have page text: did page numbers show up in the trace?
empty_pt = empty[empty.has_page_text].copy()
echo_mix = (
    empty_pt.assign(echo=np.where(
        empty_pt.page_number_echoed_in_trace, "page number(s) echoed in trace", "no page-number echo"
    ))
    .groupby(["run", "echo"], observed=True).size()
    .groupby("run", observed=True).transform(lambda s: s / s.sum())
    .unstack("echo").fillna(0)
)
share_bar(axes[1], echo_mix, "Empty answers: page numbers mentioned in trace?")
ns_e = empty_pt.groupby("run", observed=True).size().reindex(KEEP_RUNS)
axes[1].set_xticklabels([f"{r}\n(n={ns_e[r]})" for r in KEEP_RUNS])
plt.tight_layout()
plt.show()

# Cross: missed gold × echo
miss = empty_pt[empty_pt.n_gold_page > 0]
print("Empty answer BUT gold on page — does the trace still mention page numbers?")
display(
    miss.groupby(["run", "page_number_echoed_in_trace"], observed=True).size()
        .unstack("page_number_echoed_in_trace").fillna(0).astype(int)
        .rename(columns={False: "no echo", True: "echo"})
)

## 7. Example browser

Inspect concrete cases for a run and pattern.

In [ ]:
def show_examples(where: str, run: str, n: int = 3, seed: int = SEED):
    """where: one of
    - answer_in_trace_on_page
    - answer_in_trace_not_on_page
    - answer_not_in_trace
    - empty_with_gold_echo
    - empty_with_gold_no_echo
    """
    if where == "answer_in_trace_on_page":
        pool = ans[(ans.run == run) & (ans.bucket == "in trace + on page (all)")]
    elif where == "answer_in_trace_not_on_page":
        pool = ans[(ans.run == run) & (ans.bucket == "in trace, NOT on page")]
    elif where == "answer_not_in_trace":
        pool = ans[(ans.run == run) & ~ans.answer_in_trace_any]
    elif where == "empty_with_gold_echo":
        pool = empty_pt[(empty_pt.run == run) & (empty_pt.n_gold_page > 0) & empty_pt.page_number_echoed_in_trace]
    elif where == "empty_with_gold_no_echo":
        pool = empty_pt[(empty_pt.run == run) & (empty_pt.n_gold_page > 0) & ~empty_pt.page_number_echoed_in_trace]
    else:
        raise ValueError(where)

    if pool.empty:
        print("no examples for", where, run)
        return

    sample = pool.sample(min(n, len(pool)), random_state=seed)
    base = pages.set_index(["run", "report", "page"], drop=False)

    for row in sample.itertuples():
        full = base.loc[(row.run, row.report, row.page)]
        if isinstance(full, pd.DataFrame):
            full = full.iloc[0]
        page_text = lookup_page_text(row.run, row.report, row.page) or ""
        print("=" * 88)
        print(f"{row.run} | {row.report} | page {row.page} | {row.category}")
        print("answer values:", full.answer_values)
        if row.has_answer_values:
            print(f"first_frac={row.first_frac:.2f}" if pd.notna(row.first_frac) else "first_frac=nan")
            for v in full.answer_values:
                print(f"  value {v}: trace={first_match(full.reasoning, v)} page={first_match(page_text, v)}")
        else:
            print(f"empty answer | gold_on_page={row.n_gold_page} | echo={row.page_number_echoed_in_trace}")
            print("trace head:", repr((full.reasoning or "")[:240]))


show_examples("answer_in_trace_on_page", "gpt-oss medium")
show_examples("answer_in_trace_not_on_page", "DeepSeek high")
show_examples("empty_with_gold_echo", "gpt-oss low")

## Limitations

- Digit match ≠ the model used the number correctly.
- Empty-answer "echo" uses heuristic page numbers (4+ digit or thousand-separated); years and IDs can false-trigger.
- Presence on the page is necessary-ish for grounding, not proof of correct reading.
- Shares are within-run; sample sizes differ (see `n=` under bars).